Group Members: Abijeet Dhillon, Noor Fatima, Tam Nguyen

* Data Source: https://databank.worldbank.org/source/world-development-indicators#advancedDownloadOptions
* Describe the data:

# **Potential Question**

**Economic development and  Energy Resources**
* How does energy use per capita relate to GDP per capita — does energy use continue to increase as countries become wealthier, or does it level off?
* How does dependence on energy imports relate to economic growth — do countries that rely more heavily on imported energy show different growth patterns?
* Do countries with higher natural-resource rents have higher GDP per capita, or is resource dependence unrelated to economic development
* Do countries with increasing electricity access also experience higher GDP per capita growth?

**Urbanization and Energy use**
* How does urbanization correlate with CO2 emissions per capita — does it rise together or decouple in wealthier countries?
* How does urbanization relate to energy use per capita — is the relationship different between lower-income and wealthier countries?

**Economic Development and CO2**
* How does energy use per capita relate to CO2 emissions — do countries with higher renewable energy use show a different relationship?
* How does GDP per capita relate to CO2 emissions per capita — do emissions continue to increase at higher income levels or begin to level off?
* Does greater renewable energy consumption correspond to lower CO2 emissions — and is this relationship different in wealthier countries?

# **ML Question**:
* Can economic, energy, urbanization, and environmental indicators be used to predict a country's GDP per capita? ( X 2015 ​→ GDP per capita 2015)
* Can current economic and energy indicators predict future GDP per capita growth? ( X 2015​→GDP per capita growth 2016)




In [1]:
# Import python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Data collection

##Import data into colaboratory.

In [ ]:
# Connect with GG Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import the csv file
file_path = "/content/drive/MyDrive/Colab Notebooks/UoC/DATA601_Project/Data.xlsx"

df = pd.read_excel(file_path)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/UoC/DATA601_Project/Data.xlsx'

##Determine the types of data

In [ ]:
# Get information about the dataset
print("Dataset shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nNumber of unique countries:")
print(df["Country Name"].nunique())

print("\nUnique countries:")
print(df["Country Name"].unique())

## Data Cleaning/Reformating

Remove unnecessary columns (Country code and Series Code)

In [ ]:
df = df.drop(columns=["Country Code", "Series Code"])

print("Dataset shape:", df.shape)

df.head()

Reshape the data for easier analysis

In [ ]:
# Convert year to rows
df_long = df.melt(
    id_vars=["Country Name", "Series Name"],
    var_name="Year",
    value_name="Value"
)

df_long["Year"] = (
    df_long["Year"]
    .str.extract(r"(\d{4})")[0]                        # 2011 [YR2011] -> 2011
    .astype(int)                                       # Convert to int for easier filter later
)

df_long.head()

In [ ]:
# Convert indicators from rows to columns
df_reshape = df_long.pivot(
    index=["Country Name", "Year"],
    columns="Series Name",
    values="Value"
).reset_index()

df_reshape.columns.name = None                # Remove the extra label above the columns

df_reshape.head()

In [ ]:
# Dataset information after reformating
print("Dataset shape:")
print(df_reshape.shape)

print("\nNumber of countries:")
print(df_reshape["Country Name"].nunique())

print("\nData types:")
print(df_reshape.dtypes)

In [ ]:
# Checking missing values
missing_count = df_reshape.isna().sum()

missing_percent = (df_reshape.isna().mean() * 100)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

missing_summary = missing_summary.sort_values(
    "Missing Percent",
    ascending=False
)

missing_summary


We did not remove missing indicators at this stage because removing all observations with missing values could unnecessarily reduce the dataset and exclude useful information. Instead, we will handle missing values based on each analysis's requirements and the available data.

In [ ]:
# Check missing data by years
missing_data = df_reshape.isna()
missing_data["Year"] = df_final["Year"]

missing_by_year = missing_data.groupby("Year").sum()                    # Count missing values by year
missing_by_year["Total Missing"] = missing_by_year.sum(axis=1)          # Add total missing values for each year

number_indicators = len(df_reshape.columns) - 2                           # Number of indicators (not include Country Name and Year)
number_countries = df_reshape["Country Name"].nunique()                   # Number of countries
total_values = number_indicators * number_countries                     # Total values in one year

missing_by_year["Missing Percentage"] = (                               # Calculate percentage of missing values
    missing_by_year["Total Missing"] / total_values
) * 100

missing_by_year

The missing percentage is relatively stable from 2006 to 2021 (~10%) but increases sharply after 2021 (>20%). Therefore, we will focus on the 2006–2021 period, where the data coverage is more consistent.

In [ ]:
df_final = df_reshape[df_reshape["Year"].between(2006, 2021)].copy()
df_final.head()

In [ ]:
df_final.shape

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df_final.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/UoC/DATA601_Project/df_final.csv",
    index=False
)

In [ ]:
#Reload dataset
from google.colab import drive
drive.mount("/content/drive")
file_path = "/content/drive/MyDrive/Colab Notebooks/UoC/DATA601_Project/df_final.csv"
df_final = pd.read_csv(file_path)
df_final.head()

In [ ]:
# Data integrity check
df_final.dtypes

# 2. Provide summary stats for key variables and interpretation

- Correctly computes
stats (mean, median,
sd, min/max, counts)
for at least two
variables.
- Concise, accurate
interpretation of
findings

GDP per capita has a mean of ~24,559 international dollars and a median of 15,067. The mean is much higher than the median, suggesting some countries have much higher GDP per capita than others. The values range from 858 to 174,570 international dollars, showing large differences in economic development across countries.

In [ ]:
df_final['GDP per capita, PPP (constant 2021 international $)'].describe()

On average, energy use is about 2,395 kg of oil equivalent per person, while the median is about 1,395. The large standard deviation (~2,920) and wide range (from ~9 to ~21,456) show that energy consumption varies greatly across countries.

In [ ]:
df_final['Energy use (kg of oil equivalent per capita)'].describe()

On average, about 82.5% of the population has access to electricity, while the median is 99.3%. This suggests that many countries are close to 100% electricity access, while a few countries with low electricity access make the average lower. The wide range from 0.8% to 100% and the standard deviation of about 27.8% also show a large gap in electricity access across countries.

In [ ]:
df_final['Access to electricity (% of population)'].describe()

CO₂ emissions have a mean of ~4.93 tonnes per person and a median of 2.39 tonnes. The mean is higher than the median, suggesting that some countries have much higher CO₂ emissions than others. The values range from 0 to ~202.87 tonnes per person, and the standard deviation is about 8.97, showing large differences in CO₂ emissions across countries.

In [ ]:
df_final['Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)'].describe()

# 3. Visualize variable distributions

- Multiple, appropriate
visualizations (e.g.,
histograms, boxplots)
- Correct labeling, clear
presentation, good use
of libraries.

Q1: Explain your choice of plots using the five visualization components:

* Data component -- what kinds of data are you dealing with?
* Graphical component -- what kinds of plot can you use?
* Label component -- what should be on the plot axis?
* Esthetic component -- what should you plot say, and how best to do this?
* Ethical component -- Is the graph misleading, what is left out


Mentions potential biases/limitations.




In [ ]:
# GPD per capital - Histogram (4 sub-graph 2006/2011/2016/2021) and Boxplot

In [ ]:
# Average energy use by year - Line OR Energy use -Boxplot 2006/2011/2016/2021

In [ ]:
# Access to electricity (% of population) - Histogram

In [ ]:
# Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita) - Boxplot 2006/2011/2016/2021

In [ ]:
# Renewable Energy Consumption - Boxplot 2006/2011/2016/2021

In [ ]:
# Urban population % - Histogram (4 sub-graph 2006/2011/2016/2021)

# 4. Compute and visualize correlations
- Correct correlation
matrix or heatmap.
- Clear labeling and
formatting.

Q2: Choose one or two correlations and describe what the magnitude and direction of the correlation suggests about the relationship between the two variables.

In [ ]:
correlation_var = [
    "GDP per capita, PPP (constant 2021 international $)",
    "Energy use (kg of oil equivalent per capita)",
    "Access to electricity (% of population)",
    "Urban population (% of total population)",
    "Renewable energy consumption (% of total final energy consumption)",
    "Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)"
]

In [ ]:
# correlation matrix

In [ ]:
# heatmap

In [ ]:
# scatterplot - gpd vs.energy use, gdp. vs co2, urbanization vs. energy use

# 5. Discussion: Did this exploritory data analysis help you better understand your chosen dataset? If so how? Is there still parts that don't make sense?
- Thoughtful discussion
of insights gained and
dataset limitations.
- Shows critical
engagement with EDA
process.